# 장치 명령 처리와 메시지 연동


## 학습 목표

<!-- exercise-guidance:microprocessor/telegram-bot/objective -->
메신저·웹·장치 버스가 달라도 같은 명령 처리 규칙을 재사용할 수 있도록 입력·상태·응답을 나눕니다.
<!-- /exercise-guidance:microprocessor/telegram-bot/objective -->


## 실행 조건

<!-- exercise-guidance:microprocessor/telegram-bot/execution -->
기본 활동은 Python만 사용합니다. 실제 메신저·MQTT 연결은 처리 규칙을 확인한 뒤 선택합니다.
<!-- /exercise-guidance:microprocessor/telegram-bot/execution -->


## 명령과 State Transition

<!-- exercise-guidance:microprocessor/telegram-bot/activity-1 -->
### 시작 조건

입력은 /start, /led on, /led off, /led invalid, /unknown입니다. 초기 LED 상태는 off입니다.

### 진행 방법

1. handle(text,state)가 새 상태와 응답 문장을 반환하도록 작성하세요.
2. 입력 목록을 순서대로 처리하고 각 단계의 전후 상태를 기록하세요.
3. 잘못된 인수와 알 수 없는 명령은 상태를 유지하고 사용법을 반환하세요.

### 남길 결과

- 5행 State Transition 표와 함수.
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-1 -->


In [ ]:
def handle(text,state):
    if text=='/start':return state,'Commands: /led on, /led off'
    if text=='/led on':return True,'LED on'
    if text=='/led off':return False,'LED off'
    return state,'Usage: /led on | /led off'
state=False
for text in ['/start','/led on','/led off','/led invalid','/unknown']:
    next_state,reply=handle(text,state);print(text,state,next_state,reply);state=next_state


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/telegram-bot/activity-1-record -->
**입력:**

**이전 상태:**

**다음 상태:**

**응답:**

### 확인할 점

- 실제 메시지를 보내지 않고 함수 호출만으로 검사되는가?
- invalid와 unknown에서 LED 상태가 바뀌지 않는가?
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-1-record -->


## 반복 명령과 공백

<!-- exercise-guidance:microprocessor/telegram-bot/activity-2 -->
### 시작 조건

/led on을 세 번 반복하고, 앞뒤 공백이 있는 " /led off "를 추가합니다.

### 진행 방법

1. 반복 명령을 처리했을 때 상태가 유지되는지 확인하세요.
2. strip·대소문자 처리 규칙을 정한 뒤 원 함수와 비교하세요.
3. 허용하지 않을 입력도 두 개 골라 상태 보존을 검사하세요.

### 남길 결과

- 입력 6개 이상의 표와 assert 3개.
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-2 -->


In [ ]:
tests=['/led on','/led on','/led on',' /led off ','/LED ON','']
# TODO: 정규화한 처리 함수와 원본을 비교하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/telegram-bot/activity-2-record -->
**입력:**

**정규화 결과:**

**상태:**

**허용 여부:**

### 확인할 점

- 명령 표기 통일 규칙을 입력에 따라 임의로 바꾸지 않았는가?
- 반복 on이 상태를 반전시키는 toggle처럼 동작하지 않는가?
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-2-record -->


## 전송 방식과 장치 로직 분리

<!-- exercise-guidance:microprocessor/telegram-bot/activity-3 -->
원본에서 각 역할에 해당하는 함수나 줄을 표로 매핑하세요.

### 시작 조건

웹·메신저·MQTT에서 온 입력을 {request_id,device,command} 형태로 정리합니다. 같은 request_id는 다시 실행하지 않습니다.

### 진행 방법

1. 요청 ID·장치·명령을 검증한 뒤 공통 처리 함수에 전달하세요.
2. 동일 요청을 두 번 보내도 상태 변경은 한 번만 일어나도록 처리하세요.
3. 응답과 실제 상태를 기록해 요청–처리–응답 흐름을 그리세요.

### 남길 결과

- 역할 매핑 표와 가짜 전송 로그 3행.
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-3 -->


In [ ]:
seen=set();outbox=[]
def dispatch(message):
    if message['request_id'] in seen:return 'duplicate'
    seen.add(message['request_id']);outbox.append(dict(message));return 'accepted'
message={'request_id':'r1','device':'fan','command':'on'}
print(dispatch(message),dispatch(message),outbox)
# TODO: 잘못된 device·command가 seen에 들어가기 전에 검증하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/telegram-bot/activity-3-record -->
**역할:**

**원본 함수·위치:**

**테스트용 대체:**

**확인한 결과:**

### 확인할 점

- 토큰 없이 같은 처리 로직을 테스트할 수 있는가?
- 상태 갱신과 외부 전송을 독립적으로 확인했는가?
<!-- /exercise-guidance:microprocessor/telegram-bot/activity-3-record -->
